# Sortify MobileNetV2 — Reusable Colab GPU Training Notebook

Trains Sortify's 5-class waste classifier (`compost`, `glass`, `landfill`, `paper`, `plastic`) on a **Google Colab Pro GPU** using the exact `ml/scripts/train_v2.py` recipe.

### Expected Google Drive Layout
- Default folder: `MyDrive/Sortify Training/`
- Upload your prepared dataset archive (`sortify_dataset.zip` containing `train/<class>/...` and `val/<class>/...`) into `MyDrive/Sortify Training/`.
- Outputs saved back to `MyDrive/Sortify Training/runs/<RUN_TAG>/`:
  - `sortify_v2_<RUN_TAG>.pth` (CPU-portable checkpoint compatible with `backend.app.services.inference.ModelService`)
  - `training-history-<RUN_TAG>.json`
  - `run-summary-<RUN_TAG>.json` (SHA-256, best epoch, per-class val accuracy, val confusion matrix)

### What this pipeline fixes vs. v1
- Applies `ImageOps.exif_transpose` when loading images (matching `backend/app/services/inference.py`).
- Adds full `0° / 90° / 180° / 270°` right-angle rotation augmentation so sideways/landscape items do not flip classes.
- Aligns validation and mixed training resizing (`Resize((224, 224))`) with the API inference transform.

> **Note:** Keep held-out `test/` photos out of `sortify_dataset.zip` (or unused) so checkpoint selection relies strictly on `val/`.

In [ ]:
# @title 1. Check GPU Runtime & Mount Google Drive
import subprocess

import torch
from google.colab import drive

ALLOW_CPU = False  # @param {type:"boolean"}

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            text=True,
        ).strip()
    )
elif not ALLOW_CPU:
    raise RuntimeError(
        "No GPU detected! Switch Colab runtime to GPU (Runtime -> Change runtime type -> GPU)."
    )

drive.mount("/content/drive")

In [ ]:
# @title 2. Training Configuration
from pathlib import Path

RUN_TAG = "next-run"  # @param {type:"string"}
DRIVE_BASE_DIR = "/content/drive/MyDrive/Sortify Training"  # @param {type:"string"}
DATASET_ZIP = "sortify_dataset.zip"  # @param {type:"string"}
EPOCHS = 15  # @param {type:"integer"}
BATCH_SIZE = 32  # @param {type:"integer"}
LR = 1e-4  # @param {type:"number"}
WEIGHT_DECAY = 1e-4  # @param {type:"number"}
NUM_WORKERS = 2  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
USE_AMP = True  # @param {type:"boolean"}

base_dir = Path(DRIVE_BASE_DIR)
zip_path = (
    Path(DATASET_ZIP) if Path(DATASET_ZIP).is_absolute() else base_dir / DATASET_ZIP
)
run_dir = base_dir / "runs" / RUN_TAG
run_dir.mkdir(parents=True, exist_ok=True)

ckpt_out = run_dir / f"sortify_v2_{RUN_TAG}.pth"
history_out = run_dir / f"training-history-{RUN_TAG}.json"
summary_out = run_dir / f"run-summary-{RUN_TAG}.json"

print("Dataset archive:", zip_path)
print("Output run dir: ", run_dir)

In [ ]:
# @title 3. Extract Dataset to Local VM Disk & Verify Splits
import shutil
import zipfile
from collections import Counter

from torchvision import datasets

EXPECTED_CLASSES = ["compost", "glass", "landfill", "paper", "plastic"]
local_data_root = Path("/content/sortify_data")

assert zip_path.exists(), (
    f"Dataset zip not found at {zip_path}. Upload it to Google Drive first."
)
if local_data_root.exists():
    shutil.rmtree(local_data_root)
local_data_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(local_data_root)

# Handle archives that either contain train/val directly or a single nested dataset/ folder
if (local_data_root / "train").is_dir() and (local_data_root / "val").is_dir():
    DATA_DIR = local_data_root
else:
    candidates = [
        p
        for p in local_data_root.iterdir()
        if p.is_dir() and (p / "train").is_dir() and (p / "val").is_dir()
    ]
    assert len(candidates) == 1, (
        f"Expected train/ and val/ inside {zip_path}, found: {list(local_data_root.iterdir())}"
    )
    DATA_DIR = candidates[0]

train_check = datasets.ImageFolder(DATA_DIR / "train")
val_check = datasets.ImageFolder(DATA_DIR / "val")
assert train_check.classes == val_check.classes == EXPECTED_CLASSES, (
    f"Class mismatch: train={train_check.classes}, val={val_check.classes}, expected={EXPECTED_CLASSES}"
)

train_counts = Counter(train_check.classes[y] for _, y in train_check.samples)
val_counts = Counter(val_check.classes[y] for _, y in val_check.samples)
print(f"DATA_DIR: {DATA_DIR}")
print(f"Train ({len(train_check)}):", dict(sorted(train_counts.items())))
print(f"Val   ({len(val_check)}):", dict(sorted(val_counts.items())))

In [ ]:
# @title 4. Train MobileNetV2 on GPU (ml/scripts/train_v2.py recipe)
import hashlib
import json
import random
import time

import numpy as np
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def exif_pil_loader(path: str) -> Image.Image:
    with open(path, "rb") as f:
        img = Image.open(f)
        img.load()
    try:
        img = ImageOps.exif_transpose(img)
    except (TypeError, ValueError, OSError):
        pass
    return img.convert("RGB")


class RandomRightAngleRotation:
    def __init__(self, angles=(0, 90, 180, 270)):
        self.angles = angles

    def __call__(self, img: Image.Image) -> Image.Image:
        angle = random.choice(self.angles)
        return img if angle == 0 else img.rotate(angle, expand=True)


train_transform = transforms.Compose(
    [
        RandomRightAngleRotation(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomChoice(
            [
                transforms.Resize((224, 224)),
                transforms.RandomResizedCrop(
                    224, scale=(0.65, 1.0), ratio=(0.75, 1.33)
                ),
            ]
        ),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_ds = datasets.ImageFolder(
    DATA_DIR / "train", transform=train_transform, loader=exif_pil_loader
)
val_ds = datasets.ImageFolder(
    DATA_DIR / "val", transform=val_transform, loader=exif_pil_loader
)
classes = train_ds.classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
)

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
amp_enabled = USE_AMP and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)


@torch.no_grad()
def evaluate_val():
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    conf = torch.zeros(len(classes), len(classes), dtype=torch.long)
    for images, labels in val_loader:
        images, labels = (
            images.to(device, non_blocking=pin),
            labels.to(device, non_blocking=pin),
        )
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            outputs = model(images)
            loss = criterion(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        for t, p in zip(labels.cpu(), preds.cpu()):
            conf[t, p] += 1
    per_class = (conf.diag().float() / conf.sum(dim=1).clamp(min=1).float()).tolist()
    return total_loss / total, correct / total, per_class, conf.tolist()


history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = 0
best_per_class = None
best_conf = None
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = (
            images.to(device, non_blocking=pin),
            labels.to(device, non_blocking=pin),
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_ds)
    val_loss, val_acc, per_class, conf = evaluate_val()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc * 100:.2f}%"
    )
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_per_class = per_class
        best_conf = conf
        checkpoint = {
            "architecture": "mobilenet_v2",
            "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "class_names": list(classes),
            "best_epoch": best_epoch,
            "val_accuracy": best_val_acc,
        }
        torch.save(checkpoint, ckpt_out)
        print(f"  [*] Saved new best checkpoint -> {ckpt_out}")

elapsed = time.time() - start
ckpt_sha256 = hashlib.sha256(ckpt_out.read_bytes()).hexdigest()
history_out.write_text(json.dumps(history, indent=2), encoding="utf-8")
summary = {
    "run_tag": RUN_TAG,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "elapsed_seconds": round(elapsed, 2),
    "train_count": len(train_ds),
    "val_count": len(val_ds),
    "classes": classes,
    "best_epoch": best_epoch,
    "best_val_acc": best_val_acc,
    "per_class_val_acc": dict(zip(classes, best_per_class or [])),
    "val_confusion_matrix": best_conf,
    "checkpoint": str(ckpt_out),
    "checkpoint_sha256": ckpt_sha256,
}
summary_out.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(
    f"\nDone in {elapsed:.1f}s | Best Val Acc: {best_val_acc * 100:.2f}% (epoch {best_epoch})"
)
print("SHA-256:", ckpt_sha256)
for name, acc in summary["per_class_val_acc"].items():
    print(f"  {name}: {acc * 100:.2f}%")

In [ ]:
# @title 5. Plot Training Curves & Verify CPU Checkpoint Reload
import matplotlib.pyplot as plt

reloaded = torch.load(ckpt_out, map_location="cpu", weights_only=False)
assert reloaded["architecture"] == "mobilenet_v2"
assert reloaded["class_names"] == EXPECTED_CLASSES
verify_model = models.mobilenet_v2(weights=None)
verify_model.classifier[1] = nn.Linear(verify_model.last_channel, len(EXPECTED_CLASSES))
verify_model.load_state_dict(reloaded["model_state_dict"])
verify_model.eval()
dummy_out = verify_model(torch.zeros(1, 3, 224, 224))
assert dummy_out.shape == (1, len(EXPECTED_CLASSES))
print("CPU reload verification passed:", ckpt_out.name)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs_x = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs_x, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_x, history["val_loss"], label="Val Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(
    epochs_x, [a * 100 for a in history["val_acc"]], label="Val Acc (%)", color="green"
)
axes[1].set_title(f"Validation Accuracy (Best: {best_val_acc * 100:.2f}%)")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()